# Frame-window metric comparison
Run all cells to score saved predictions, then compare methods across frame counts. Each point is an equal-weight mean over windows with status `ok`. K is the number of input frames. M counts skipped frames within a window. Start stride controls the separation between window starts.

In [ ]:
from pathlib import Path
import numpy as np
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments/score_windows.py").is_file())
DATASET, SPLIT = "eds", "test"
EXPERIMENTS = {2: "windows_k2", 5: "windows_k5", 10: "windows_k10"}
M, START_STRIDE = 0, 50
sys.path.insert(0, str(ROOT / "benchmarks"))
import run_metric as benchmark
METRICS = list(benchmark.UNITS)  # All metrics supported by score_windows.py
BENCHMARK_METHODS = list(benchmark.v2.old.METHODS)
METHODS = []  # Empty discovers available methods for each experiment.
SEQUENCES = []  # Empty evaluates all sequences in the split.
WORKERS, DEVICE = 16, "cpu"
REPORTS = {}

## Compute metrics
Use the Rails kernel with the benchmark dependencies. Completed window scores are reused when the scorer's resume checks pass. The final line of each invocation identifies its report directory.

In [ ]:
REPORTS.clear()
for k, experiment in EXPERIMENTS.items():
    for metric in METRICS:
        command = [sys.executable, "-u", "experiments/score_windows.py", DATASET,
                   "--split", SPLIT, "--experiment-name", experiment, "--k", str(k),
                   "--m", str(M), "--start-stride", str(START_STRIDE),
                   "--metric", metric, "--workers", str(WORKERS), "--device", DEVICE]
        if METHODS:
            command.extend(["--methods", *METHODS])
        if SEQUENCES:
            command.extend(["--sequences", *SEQUENCES])
        print(f"Computing {metric} for K={k}", flush=True)
        last = ""
        with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, text=True) as proc:
            for line in proc.stdout:
                print(line, end="", flush=True)
                if line.strip():
                    last = line.strip()
            status = proc.wait()
        folder = Path(last)
        if not folder.is_absolute() or not all((folder / f).is_file() for f in ("summary.csv", "per_window.csv")):
            print(f"Unavailable: K={k}, {metric}. See scorer output above.", flush=True)
            continue
        REPORTS[k, metric] = folder
        if status:
            print("Scoring errors recorded. Inspect per-window status before interpreting means.")

## Means and coverage
Inspect valid and expected window counts alongside each mean. Different methods can have different valid windows even when their counts match.

In [ ]:
if not REPORTS:
    raise RuntimeError('No reports. Check the experiment names and scorer output above.')
summary = pd.concat([pd.read_csv(p / "summary.csv") for p in REPORTS.values()], ignore_index=True)
windows = pd.concat([pd.read_csv(p / "per_window.csv") for p in REPORTS.values()], ignore_index=True)
display(summary[["k", "metric", "method", "mean", "unit", "n_valid", "n_total", "status_counts"]]
        .sort_values(["metric", "method", "k"]))

## Model means by K
Dark, medium and light blue denote the first three distinct scores. Ties share a color. Values are the saved window means. For descriptive metrics the color ranks distance from the displayed GT mean. For the count ratio it ranks distance from 1. For error metrics smaller means rank first. GT is excluded from ranking. A dash denotes an unavailable scalar; see coverage above. ISI distribution is shown separately as a curve.

In [ ]:
model_order = METHODS or BENCHMARK_METHODS
scalar_metrics = [m for m in METRICS if m != "isi_distribution"]
palette = {1: ("#18548c", "white"), 2: ("#82b7df", "#10263b"), 3: ("#d9ebfa", "#10263b")}

def top_three(column):
    scores = column.drop(index="ground_truth").copy()
    if column.name in benchmark.DESCRIPTIVE:
        scores = (scores - column.at["ground_truth"]).abs()
    elif column.name == "event_count_ratio":
        scores = (scores - 1).abs()
    ranks = scores.rank(method="dense")
    styles = pd.Series("", index=column.index)
    for model, rank in ranks.dropna().items():
        if rank <= 3:
            bg, fg = palette[int(rank)]
            styles[model] = f"background-color: {bg}; color: {fg}"
    return styles

for k in sorted(EXPERIMENTS):
    table = (summary.loc[summary.k == k].pivot(index="method", columns="metric", values="mean")
             .reindex(index=["ground_truth", *model_order], columns=scalar_metrics))
    from html import escape
    from IPython.display import HTML
    styles = table.apply(top_three, axis=0)
    header = "".join(f"<th>{escape(col)}</th>" for col in table.columns)
    body = []
    for method, row in table.iterrows():
        cells = []
        for metric, value in row.items():
            label = "—" if pd.isna(value) else f"{value:.5g}"
            cells.append(f'<td style="{styles.at[method, metric]}">{label}</td>')
        body.append(f"<tr><th>{escape(method)}</th>{''.join(cells)}</tr>")
    display(HTML(f'<div style="overflow-x:auto"><table><caption>K = {k} | Mean over valid windows</caption>'
                 f"<thead><tr><th>Method</th>{header}</tr></thead><tbody>{''.join(body)}</tbody></table></div>"))

## Compare K within each metric
One line per method. Gaps indicate an unavailable mean. A dotted reference at 1 indicates the target for event-count ratio.

In [ ]:
ks = sorted(EXPERIMENTS)
methods = sorted(summary.method.unique())
colors = dict(zip(methods, plt.get_cmap("tab20").colors))
for metric in [m for m in METRICS if m != 'isi_distribution']:
    rows = summary.loc[summary.metric == metric]
    if rows.empty:
        continue
    fig, ax = plt.subplots(figsize=(9, 4))
    for method, group in rows.groupby("method"):
        values = group.set_index("k")["mean"].reindex(ks)
        if values.notna().any():
            if method == "ground_truth":
                ax.plot(ks, values, "k--s", linewidth=2.5, label="GT (recorded)", zorder=5)
            else:
                ax.plot(ks, values, marker="o", color=colors[method], label=method)
    if metric == "event_count_ratio":
        ax.axhline(1, linestyle=":", color="black", label="Target ratio = 1")
    elif metric not in benchmark.DESCRIPTIVE:
        ax.axhline(0, linestyle=":", color="grey", label="Error target = 0")
    ax.set(xticks=ks, xlabel="Input frames per window (K)", ylabel=rows.unit.iloc[0], title=metric)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
    else:
        ax.text(.5, .5, "No valid scalar means", ha="center", transform=ax.transAxes)
    ax.grid(axis="y", alpha=.25)
    plt.tight_layout()
    plt.show()

## ISI distribution by K
Each curve is the mean of the per-window empirical cumulative distributions over windows with at least one interval. The grid spans 0 to the longest evaluated window in milliseconds.

In [ ]:
for k in sorted(EXPERIMENTS):
    folder = REPORTS.get((k, "isi_distribution"))
    if folder is None:
        continue
    rows = windows[(windows.k == k) & (windows.metric == "isi_distribution") & (windows.status == "ok")]
    if rows.empty:
        continue
    upper = max(1e-6, rows.duration_s.max() * 1000)
    grid = np.r_[0, np.geomspace(min(1e-6, upper), upper, 160)]
    fig, ax = plt.subplots(figsize=(8, 3.8))
    for method, group in rows.groupby("method"):
        total = np.zeros_like(grid)
        count = 0
        for rel in group.array_path.dropna():
            with np.load(folder / rel) as data:
                intervals = np.sort(data["isi_ms"])
            if intervals.size:
                total += np.searchsorted(intervals, grid, side="right") / intervals.size
                count += 1
        if count:
            gt = method == "ground_truth"
            ax.plot(grid, total/count, "--" if gt else "-", color="black" if gt else colors[method],
                    label="GT (recorded)" if gt else method, zorder=5 if gt else 2)
    ax.set_xscale("symlog", linthresh=0.001)
    ax.set(xlabel="ISI (ms)", ylabel="Mean cumulative fraction", title=f"ISI distribution | K = {k}")
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.show()

## Inspect individual windows
Change the filters to inspect individual values and missing or partial scores.

In [ ]:
SHOW_K, SHOW_METRIC = min(EXPERIMENTS), METRICS[0]
SHOW_METHOD, SHOW_SEQUENCE = None, None
selected = windows[(windows.k == SHOW_K) & (windows.metric == SHOW_METRIC)]
if SHOW_METHOD:
    selected = selected[selected.method == SHOW_METHOD]
if SHOW_SEQUENCE:
    selected = selected[selected.sequence == SHOW_SEQUENCE]
display(selected.sort_values(["sequence", "first", "method"]))